In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### Problema:
  - Ou seja, os DEGs "de cada cluster" são majoritariamente a assinatura do polo, não do cluster. 
  - Isso reproduz exatamente a estrutura binária Program-1/Program-2 que você viu em N=3.
  - Para recuperar o que é próprio de cada um, calculei o resíduo (LFC do cluster menos a média do grupo) — é daí que sai a anotação abaixo.

### Alerta sério: o eixo primário parece técnico, não biológico. A composição por biotipo de transcrito:

| 	                | UP: codificante	| UP: antisenso+lncRNA |	DOWN: lncRNA/pseudogene  |
|-------------------|-----------------|----------------------|---------------------------|
| C6/7/9/10         | ~91%	| ~0,4%	| ~82% |
| C3                | 27%	| 66%	| 43% |

### Nenhum programa biológico real se separa perfeitamente por classe de anotação. 

Esse padrão — HOX-antisenso, -AS1, clones AC/AL/AP, pseudogenes (RPL9P28, NARS1P2, PSMD4P1, BRD7P3) de um lado; codificantes do outro; KCNQ1OT1 a −9 e DLEU1/2 a −5 — é a assinatura clássica de diferença em integridade de RNA / fração intrônica / profundidade de biblioteca. Isso recomenda revisitar a caracterização anterior do Program-1 como "programa regulatório lncRNA/HOX-antisenso": pode ser em boa parte artefato. Vale checar RIN, fração exônica e lote antes de seguir.


### Combat
  - Para uma coorte TCGA eu ficaria no R. 


### Erro no conceito de malignidade

Segundo ponto, sobre o desenho: o compartimento maligno não tem LFC tumor vs. controle definido. Não há células malignas em pâncreas normal — θ_malignant nos 21 controles vai para ~0, e qualquer LFC ali é razão entre um posterior real e ruído do prior. O script exclui esse compartimento do teste e oferece o contraste defensável no lugar: ductal maligno (tumor) vs. ductal normal (controle).

### Prism
  - A tabela ComBat não serve como input do BayesPrism.
  - Tambémm não usar CPM
  - Temos que trabalhar sobre contagens --> deconvolução single-cell --> unir os resultados

### Sequenciamento TCGA-PAAD e CPTAC3

- TCGA-PAAD foi sequenciado com biblioteca não-direcional; 
- CPTAC3 usa protocolo stranded 

Confirme no seu metadado, mas as bibliotecas do TCGA são efetivamente não-stranded — o kit usado permitiria strand-specificity, mas foi aplicado de modo que as bibliotecas não a preservam. 

Em biblioteca não-direcional você não consegue atribuir uma leitura ao gene certo quando existe um gene sobreposto na fita oposta. O efeito é enorme e restrito a uma classe: 
  - cerca de 10% de todos os genes e 2,5% dos protein-coding têm diferença de duas vezes ou mais na expressão estimada quando a informação de fita é ignorada.


#### Question

https://www.biostars.org/p/113679/

I want more details about Strand-specific vs. Non strand-specific protocols in RNAseq. I know that the benefit of strand-specific is to know whether the read originated from the +ve or - ve strand, also it helps identify antisense RNA, and I know it costs more than non-strand-specific. So could you please give me more information about this protocol. I am going to do RNA-seq analysis for human and mouse to detect antisense.

The strand specific library should only be slightly more expensive than the non-strand specific library from what I remember. So I always prefer the stranded library. What kind of information do you want? The procedure of the protocol? Or the type of strand specific library?

https://www.seqanswers.com/forum/applications-forums/rna-sequencing/17157-question-about-stranded-rna-seq?t=20489

Non stranded protocols convert mRNA (single stranded) into cDNA (double stranded) before library prep.

When you do this, you are then unable to tell the strand of the genomic DNA from which the mRNA was originally transcribed. The data obtained from the sequencer could feasible come from either cDNA strand.

There are two main methods of creating a stranded library. 

- Firstly, you could adapt the small RNA protocol where adapters are ligated directly to the RNA one at a time. That way you get different adapters on the 5' and 3' end and only sequence in one direction.
- A second method is to incorporate dUTP into the second strand cDNA synthesis. At the end of library prep, you then treat with USER which will digest the Uracil tagged strand, leaving you with only library from the first cDNA strand.
- The third method I have seen (commercial kit) is anneal 3' extension-blocked random oligos to the 1st strand cDNA and extend the cDNA into specific adapter sequence.

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> TCGA-PAAD Tumor
>>> case Tumor
	DEGs 20484
		Up (#12140)
		Dw (#8344)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    13
1                      IG_C_pseudogene     3
2                            IG_J_gene     9
3                      IG_J_pseudogene     1
4                            IG_V_gene   127
5                      IG_V_pseudogene    48
6                              Mt_tRNA    17
7                                  TEC   148
8                            TR_C_gene     6
9                            TR_D_gene     1
10                           TR_J_gene     8
11                           TR_V_gene    45
12                     TR_V_pseudogene     4
13                              lncRNA  4162
14                               miRNA   139
15                            misc_RNA    36
16              polymorphic_pseudogene     4


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'


verbose=True
cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)


-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"


### Calc expression

 - calc_file_expression_tumor_normal_gtex()
   - get_dic_expression_tumor_and_normal()
     - get_filtered_tables()
     - get_table_given_fileID()
   - prepare_normal_tumor_tables()

  
#### Tables in: root_disease / lfc

In [8]:
cbio.root_disease, cbio.root_lfc, cbio.filename_demo

(PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc'),
 PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/clinical_and_demographics_for_paad_cptac_2021.tsv'))

### Get cases, subtypes and clin_demo tables

In [9]:
verbose=False
force=False

PROG_ID = 'TCGA'
psi_id = 'SKCM'
psi_id = 'BRCA'
psi_id = 'PAAD'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

df_cases, df_subt, df_clin_demo, df_case_bar = cbio.get_cases_and_subtypes(batch_size=200, force=force, verbose=verbose)

df_cases.shape, df_clin_demo.shape, df_case_bar.shape

((170, 27), (140, 20), (140, 2))

In [10]:
verbose=True

dic_tumor, dic_normal = cbio.get_dic_expression_tumor_and_normal(verbose=verbose)

Table opened ((170, 27)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/cases_for_PAAD.tsv'
Table opened ((5953, 14)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples/samples_for_CPTAC3_PAAD_Pancreas_subtype_other_tumor_other_tissue_other.tsv'
Table opened ((4897, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations/mutations_anal_for_CPTAC3_PAAD_Pancreas_subtype_other_tumor_other_tissue_other.tsv'
Table opened ((170, 27)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/cases_for_PAAD.tsv'
Table opened ((5953, 14)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples/samples_for_CPTAC3_PAAD_Pancreas_subtype_other_tumor_other_tissue_other.tsv'
Table opened ((4897, 26)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations/mutations_anal_for_CPTAC3_PAAD_Pancreas_subtype_other_tumor_other_tissue_other.tsv'
There are 49 tumor and 20 normal Gene Expression tables
.Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc/Gene

In [16]:
cbio.root_lfc

PosixPath('/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc')

In [17]:
files = [x for x in os.listdir(cbio.root_lfc) if x.startswith('Gene_Expression_Quantification_tumor_for_')]
files[:3]

['Gene_Expression_Quantification_tumor_for_PAAD_case_465c6020-db87-4c81-a52c-1b23a78d5adc_file_4f3b4b56-f9a8-4fef-9731-7b114d54c016.tsv',
 'Gene_Expression_Quantification_tumor_for_PAAD_case_1309999e-853f-4f99-84a2-1095e674f5ae_file_b1ea7f49-c929-41c5-89eb-acab9db9c6e4.tsv',
 'Gene_Expression_Quantification_tumor_for_PAAD_case_0b2f0141-88e1-4117-a668-f284ec7670ff_file_274b2655-ae32-419f-8494-6992d2a4a9fb.tsv']

In [20]:
fname = cbio.root_lfc / files[0]

df = pdreadcsv(files[0], cbio.root_lfc, verbose=True)
df.rename(columns={'counts': 'stranded_first'}, inplace=True)

Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc/Gene_Expression_Quantification_tumor_for_PAAD_case_465c6020-db87-4c81-a52c-1b23a78d5adc_file_4f3b4b56-f9a8-4fef-9731-7b114d54c016.tsv'


In [21]:
df

,geneid,symbol,biotype,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,ENSG00000000003,TSPAN6,protein_coding,821,0,821,17.235,6.215,5.889
1,ENSG00000000005,TNMD,protein_coding,3,2,1,0.194,0.070,0.066
2,ENSG00000000419,DPM1,protein_coding,1120,9,1112,88.357,31.861,30.194
3,ENSG00000000457,SCYL3,protein_coding,1058,199,1252,14.637,5.278,5.002
4,ENSG00000000460,C1orf112,protein_coding,405,342,493,6.460,2.329,2.208
...,...,...,...,...,...,...,...,...,...
60655,ENSG00000288669,AC008763.4,protein_coding,1,0,1,0.024,0.009,0.008
60656,ENSG00000288670,AL592295.6,lncRNA,192,0,204,10.117,3.648,3.457
60657,ENSG00000288671,AC006486.3,protein_coding,0,0,0,0.000,0.000,0.000
60658,ENSG00000288674,AL391628.1,protein_coding,7,0,7,0.069,0.025,0.024


### TCGA-PAAD

In [22]:
verbose=False
force=False

PROG_ID = 'TCGA'
psi_id = 'SKCM'
psi_id = 'BRCA'
psi_id = 'PAAD'

cbio.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

df_cases, df_subt, df_clin_demo, df_case_bar = cbio.get_cases_and_subtypes(batch_size=200, force=force, verbose=verbose)

df_cases.shape, df_clin_demo.shape, df_case_bar.shape

((185, 27), (184, 43), (184, 2))

In [24]:
cbio.root_lfc

PosixPath('/home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc')

In [23]:
files = [x for x in os.listdir(cbio.root_lfc) if x.startswith('Gene_Expression_Quantification_tumor_for_')]
files[:3]

['Gene_Expression_Quantification_tumor_for_PAAD_case_ce9e0431-19a1-4ea9-85b0-44a9e9792e03_file_789ebf62-b31b-4ee1-86d5-b33b07d8bb33.tsv',
 'Gene_Expression_Quantification_tumor_for_PAAD_case_c53328b4-0930-4e6a-a648-283b5565295e_file_c7be34ee-0d68-47fa-bea9-601cc5da20cb.tsv',
 'Gene_Expression_Quantification_tumor_for_PAAD_case_396ff766-a383-4f42-bce3-f2f32b7f1151_file_f8ed2164-dbe4-4eb7-a7c6-c123e911e50a.tsv']

In [25]:
fname = cbio.root_lfc / files[0]

df = pdreadcsv(files[0], cbio.root_lfc, verbose=True)
df.rename(columns={'counts': 'stranded_first'}, inplace=True)

Table opened ((60660, 9)) at '/home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/Gene_Expression_Quantification_tumor_for_PAAD_case_ce9e0431-19a1-4ea9-85b0-44a9e9792e03_file_789ebf62-b31b-4ee1-86d5-b33b07d8bb33.tsv'


In [26]:
df

,geneid,symbol,biotype,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,ENSG00000000003,TSPAN6,protein_coding,1260,618,642,24.857,7.423,8.132
1,ENSG00000000005,TNMD,protein_coding,121,64,57,7.336,2.191,2.400
2,ENSG00000000419,DPM1,protein_coding,1299,674,627,96.304,28.760,31.508
3,ENSG00000000457,SCYL3,protein_coding,364,304,278,4.732,1.413,1.548
4,ENSG00000000460,C1orf112,protein_coding,164,184,219,2.458,0.734,0.804
...,...,...,...,...,...,...,...,...,...
60655,ENSG00000288669,AC008763.4,protein_coding,0,0,0,0.000,0.000,0.000
60656,ENSG00000288670,AL592295.6,lncRNA,114,51,65,5.645,1.686,1.847
60657,ENSG00000288671,AC006486.3,protein_coding,0,0,0,0.000,0.000,0.000
60658,ENSG00000288674,AL391628.1,protein_coding,4,3,1,0.037,0.011,0.012


### Problema: standed and non-stranded sequencing

Antes de escolher o método: o efeito entre TCGA e CPTAC3 provavelmente não é um batch effect no sentido usual, e isso muda a resposta.

TCGA-PAAD foi sequenciado com biblioteca não-direcional; CPTAC3 usa protocolo stranded (confirme no seu metadado, mas as bibliotecas do TCGA são efetivamente não-stranded — o kit usado permitiria strand-specificity, mas foi aplicado de modo que as bibliotecas não a preservam). Em biblioteca não-direcional você não consegue atribuir uma leitura ao gene certo quando existe um gene sobreposto na fita oposta. O efeito é enorme e restrito a uma classe: cerca de 10% de todos os genes e 2,5% dos protein-coding têm diferença de duas vezes ou mais na expressão estimada quando a informação de fita é ignorada. 
Bioinformatics Answers
nih

Isso é a explicação mecanística mais provável do eixo técnico que apareceu na sua clusterização — HOX-antisenso, -AS1, -AS3/4, lncRNAs de clone, KCNQ1OT1. São exatamente os transcritos antisenso sobrepostos. Um ComBat "resolve" isso no sentido de alinhar as médias, mas é um deslocamento aditivo aplicado a um viés que é multiplicativo, gene-específico e dependente da expressão do parceiro na fita oposta. Fica melhor, não fica certo.

### Ordem de ataque

1. Checar confundimento antes de qualquer coisa. Este é o risco maior.
  - TCGA-PAAD tem só ~4 normais de pâncreas; CPTAC3 tem dezenas. 
  - Se os seus 21 controles vierem quase todos do CPTAC3, então estudo ≈ grupo, 
  - e o ComBat_seq vai remover justamente o sinal tumor-vs-normal que você quer medir. 
  - Se a tabela cruzada mostrar isso, a resposta certa é não corrigir e sim estratificar.

2. Se strandedness for a causa, a correção certa é na quantificação, não na matriz. Remover os genes com sobreposição antisenso resolve o problema na origem em vez de mascará-lo. Custa ~10% dos genes, e para deconvolução isso é irrelevante — o select.gene.type("protein_coding") do BayesPrism já derruba a maior parte deles de qualquer forma.

3. Considerar não corrigir nada. O BayesPrism deconvolve cada amostra de forma independente: θ da amostra i não depende da amostra j. Um deslocamento no nível de estudo não se propaga como se propagaria numa clusterização conjunta. O que resta de risco é o ajuste ao φ compartilhado, e é aí que o filtro de genes atua. Depois, o estudo entra como covariável no modelo de DE (~ study + group, que o script anterior já suporta pela coluna batch) — corrigir os dados e modelar o estudo é dupla correção.

4. Se for corrigir mesmo: ComBat_seq, nunca ComBat. Devolve contagens inteiras a partir de um modelo binomial negativo, compatível com o BayesPrism. Sempre com group= preenchido.

In [28]:
root_check = cbio.root_src / 'check'
os.listdir(root_check)

['combat_seq.R', 'batch_diag.py']

In [29]:
if str(root_check) not in sys.path:
    sys.path.append(str(root_check))


In [30]:
d = pd.read_csv(cbio.root_mprog_lfc / "metadata_tumor_and_normal.tsv", sep="\t")
design = (d.rename(columns={"cols": "sample", "program": "study", "condition": "group"})
            [["sample", "study", "group"]]
            .assign(group=lambda x: x.group.replace({"normal": "control"})))
design.to_csv(root_check / "design.csv", index=False)
design

,sample,study,group
0,T-C3L-02890,CPTAC3,tumor
1,T-C3L-03635,CPTAC3,tumor
2,T-C3L-02701,CPTAC3,tumor
3,T-C3L-04072,CPTAC3,tumor
4,T-C3L-00589,CPTAC3,tumor
...,...,...,...
147,N-C3N-02996,CPTAC3,control
148,N-C3L-02606,CPTAC3,control
149,N-C3N-03173,CPTAC3,control
150,N-C3N-02696,CPTAC3,control


In [32]:
root_check

PosixPath('/home/flavio/uv/perturb_agent/src/check')

In [35]:
cbio.root_mprog_lfc

PosixPath('/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc')

### O que é o V de Cramér

O V de Cramér mede **quão fortemente duas variáveis categóricas estão associadas**, numa escala de 0 a 1. Aqui as duas variáveis são `study` (TCGA vs CPTAC3) e `group` (tumor vs controle).

$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r,c) - 1)}}$$

onde $\chi^2$ é a estatística qui-quadrado da tabela cruzada, $n$ o total de amostras, e $r$, $c$ o número de linhas e colunas.

**Por que não usar o qui-quadrado direto:** $\chi^2$ cresce com o tamanho da amostra. Duplique todas as células de uma tabela e o padrão de associação é idêntico, mas o $\chi^2$ dobra. Isso o torna inútil para comparar desenhos experimentais de tamanhos diferentes. O V divide por $n$ e pelo tamanho da tabela, produzindo um número comparável entre estudos.

**Interpretação nas pontas:**
- **V = 0** — as variáveis são independentes. Cada estudo contribui com tumores e controles na mesma proporção. Saber de qual coorte a amostra veio não diz nada sobre ser tumor ou não.
- **V = 1** — associação perfeita. Saber a coorte determina o grupo. Estudo e biologia são a *mesma* variável, com dois nomes.

E é exatamente por isso que a métrica cabe aqui: **confundimento é literalmente associação entre a variável técnica e a variável biológica.** Quando V se aproxima de 1, não existe correção possível — nenhum método consegue separar duas coisas que os dados registram como uma só.

### Escala, com os seus dados no meio

| Cenário | Tabela | V |
|---|---|---|
| Balanceado perfeito | 35/41 vs 35/41 | 0,000 |
| Desbalanço leve | 15/60 vs 6/71 | 0,154 |
| **Seu caso** | **20/49 vs 1/82** | **0,373** |
| Quase confundido | 21/20 vs 0/111 | 0,625 |
| Confundido total | 21/0 vs 0/131 | 0,947 |

As referências convencionais para tabela 2×2 (adaptadas de Cohen) são: 0,1 pequeno, 0,3 médio, 0,5 grande.

Um detalhe de implementação: no código eu somo 0,5 a todas as células antes do cálculo (correção de Haldane-Anscombe). Sem ela, o V do seu caso seria 0,382 — diferença desprezível, mas a correção evita instabilidade numérica quando alguma célula é muito pequena, que é precisamente a sua situação.

### O que o V *não* viu — e isso importa

Aqui preciso ser direto sobre uma limitação do que eu coloquei no seu código.

V = 0,373 é uma associação "média" pelas convenções. Se o veredito dependesse só do V, a função teria devolvido `OK`. **O que disparou o alerta foi a outra condição da regra:** `min_cell < 5`.

```python
elif min_cell < 5 or cramers_v > 0.7:
    verdict = "PARCIAL"
```

O V mede a *força* da associação de forma agregada. Ele não enxerga que uma célula específica tem n=1. E o seu problema real não é a força da associação — é que **não existe informação suficiente numa das células para estimar coisa alguma**. Com um único controle no TCGA, você não tem variância, não tem como testar a suposição de aditividade do ComBat, não tem contraste interno.

São dois defeitos diferentes de um desenho, e o V só detecta um deles. Trate-o como um resumo útil para reportar no methods, não como o critério de decisão. O critério de decisão é olhar a tabela cruzada — que é por isso que a função a imprime inteira antes de qualquer número.

In [37]:
!python /home/flavio/uv/perturb_agent/src/check/batch_diag.py check_confounding --design /home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/metadata_tumor_and_normal.tsv

loading ...
Coluna de estudo: usando 'program'
Coluna de grupo: usando 'condition'
Tabela cruzada estudo x grupo:
condition  control  tumor
program                  
CPTAC3          20     49
TCGA             1     82
Cramér's V = 0.375 | célula mínima = 1 | veredito: PARCIAL
ComBat_seq vai encolher o efeito biológico. Prefira não corrigir e usar ~ study + group no modelo.
